<a href="https://colab.research.google.com/github/sipy/su5-phase-field/blob/main/Heavy_Nuclei_Crystalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import math
import time

# 1. HARDWARE SETUP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64)
print(f"Hardware: {device.type.upper()} | FUSION CRYSTALLIZATION MODE")

N = 128
dx = 0.08
mask = torch.zeros(1, 4, N, N, N, device=device)
mask[:, :, 2:-2, 2:-2, 2:-2] = 1.0
vacuum = torch.tensor([1.0, 0.0, 0.0, 0.0], device=device).view(1, 4, 1, 1, 1)

# 2. THE B=7 HEAVY NUCLEUS SEED
print("Generating B=7 (Lithium/Beryllium) Topological Seed...")
X, Y, Z = torch.meshgrid(torch.arange(N, device=device),
                         torch.arange(N, device=device),
                         torch.arange(N, device=device), indexing='ij')

cx, cy, cz = N // 2, N // 2, N // 2
rx = (X - cx) * dx
ry = (Y - cy) * dx
rz = (Z - cz) * dx
r = torch.sqrt(rx**2 + ry**2 + rz**2) + 1e-8

theta = torch.acos(rz / r)
phi_angle = torch.atan2(ry, rx)

R_knot = 1.8 # Slightly larger starting radius to fit all 7 charges
f = math.pi * torch.exp(-r / R_knot)

# The B=7 wrapping
phi0 = torch.cos(f)
phi1 = torch.sin(f) * torch.sin(theta) * torch.cos(7 * phi_angle)
phi2 = torch.sin(f) * torch.sin(theta) * torch.sin(7 * phi_angle)
phi3 = torch.sin(f) * torch.cos(theta)

phi = torch.stack([phi0, phi1, phi2, phi3], dim=0).unsqueeze(0)

# SYMMETRY BREAKING: We inject 5% random noise.
# Without this, the knot might get stuck as a perfect donut. The noise allows
# the topology to organically fracture and crystallize into the 12-faced Dodecahedron.
noise = torch.randn_like(phi) * 0.05
phi = phi + noise

phi = F.normalize(phi, p=2, dim=1)
phi = phi * mask + vacuum * (1.0 - mask)
phi.requires_grad_(True)

# 3. PHYSICS ENGINE (Identical to previous runs)
def get_derivatives_4th_order(phi_tensor, dx_val):
    k_vals = [1.0/12.0, -8.0/12.0, 0.0, 8.0/12.0, -1.0/12.0]
    kernel = torch.tensor(k_vals, device=device) / dx_val
    kx = kernel.view(1, 1, 1, 1, 5).repeat(4, 1, 1, 1, 1)
    ky = kernel.view(1, 1, 1, 5, 1).repeat(4, 1, 1, 1, 1)
    kz = kernel.view(1, 1, 5, 1, 1).repeat(4, 1, 1, 1, 1)

    pad_x = F.pad(phi_tensor, (2, 2, 0, 0, 0, 0), mode='replicate')
    dx_phi = F.conv3d(pad_x, kx, groups=4)
    pad_y = F.pad(phi_tensor, (0, 0, 2, 2, 0, 0), mode='replicate')
    dy_phi = F.conv3d(pad_y, ky, groups=4)
    pad_z = F.pad(phi_tensor, (0, 0, 0, 0, 2, 2), mode='replicate')
    dz_phi = F.conv3d(pad_z, kz, groups=4)
    return torch.stack([dx_phi, dy_phi, dz_phi], dim=0)

def compute_skyrme_energy(phi_tensor, dx_val, f_pi=1.0, e=0.4, m_pi=0.05):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    G = torch.einsum('iczyx, jczyx -> ijzyx', dphi[:, 0], dphi[:, 0])
    E2 = G[0,0] + G[1,1] + G[2,2]
    TrG_sq = E2**2
    Tr_G2 = sum(G[i,j] * G[j,i] for i in range(3) for j in range(3))
    E4 = 0.5 * (TrG_sq - Tr_G2)
    E_mass = (m_pi**2) * (1.0 - phi_tensor[0, 0])
    energy_density = (f_pi**2 / 16.0) * E2 + (1.0 / (32.0 * e**2)) * E4 + E_mass
    return torch.sum(energy_density) * (dx_val**3)

def compute_winding_number(phi_tensor, dx_val):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    vectors = torch.stack([phi_tensor[0], dphi[0,0], dphi[1,0], dphi[2,0]], dim=0)
    matrix_field = vectors.permute(2, 3, 4, 0, 1)
    det_density = torch.linalg.det(matrix_field)
    return torch.sum(det_density) * (dx_val**3) / (2 * math.pi**2)

# 4. THE RELAXATION LOOP
with torch.no_grad():
    print(f"Initial Noisy Seed Winding B: {compute_winding_number(phi, dx).item():.5f}\n")

optimizer = torch.optim.Adam([phi], lr=0.002)
start_time = time.time()

# 1000 steps to let the 12 faces form
for step in range(1001):
    optimizer.zero_grad()
    energy = compute_skyrme_energy(phi, dx)
    energy.backward()
    phi.grad.clamp_(-0.02, 0.02)
    optimizer.step()

    with torch.no_grad():
        phi.data = phi.data * mask + vacuum * (1.0 - mask)
        phi.copy_(F.normalize(phi, p=2, dim=1))

    if step % 100 == 0:
        with torch.no_grad():
            B = compute_winding_number(phi, dx)
            print(f"Step {step:4d} | Energy: {energy.item():.4f} | System Winding B: {B.item():.5f}")

print("\nCrystallization Complete.")

Hardware: CUDA | FUSION CRYSTALLIZATION MODE
Generating B=7 (Lithium/Beryllium) Topological Seed...
Initial Noisy Seed Winding B: 6.80324

Step    0 | Energy: 1375.8109 | System Winding B: 6.80593
Step  100 | Energy: 365.1351 | System Winding B: 6.88180
Step  200 | Energy: 348.4224 | System Winding B: 6.85079
Step  300 | Energy: 338.2881 | System Winding B: 6.79617
Step  400 | Energy: 330.8780 | System Winding B: 6.71966
Step  500 | Energy: 325.5081 | System Winding B: 6.62083
Step  600 | Energy: 321.4590 | System Winding B: 6.51336
Step  700 | Energy: 318.6672 | System Winding B: 6.41340
Step  800 | Energy: 316.8711 | System Winding B: 6.32511
Step  900 | Energy: 315.5939 | System Winding B: 6.23748
Step 1000 | Energy: 314.3762 | System Winding B: 6.14423

Crystallization Complete.


In [2]:
import numpy as np
import plotly.graph_objects as go
from skimage import measure

print("Extracting 3D geometry from the GPU...")

# We extract the phi_0 component. The surface where phi_0 = -0.5
# represents the "shell" of the topological knot.
with torch.no_grad():
    # Move the tensor from GPU to CPU, and convert to numpy
    volume = phi[0, 0, :, :, :].cpu().numpy()

# Use Marching Cubes to find the 3D surface mesh
verts, faces, normals, values = measure.marching_cubes(volume, level=-0.5)

print("Rendering Platonic Solid...")

# Create the interactive 3D plot
fig = go.Figure(data=[go.Mesh3d(
    x=verts[:, 0],
    y=verts[:, 1],
    z=verts[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    opacity=0.8,
    colorscale='Viridis',
    intensity=verts[:, 2], # Color based on Z-axis height
    flatshading=True       # This highlights the sharp geometric faces!
)])

fig.update_layout(
    title='B=7 Topological Nucleus (Isosurface)',
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='rgb(20, 20, 20)' # Dark space background
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

Extracting 3D geometry from the GPU...
Rendering Platonic Solid...
